In [4]:
!pip install evaluate

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel, Features
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig
import evaluate
import torch
import numpy as np
import time, os

In [6]:
#  mount google drive
from google.colab import drive
drive.mount('/content/drive')

dataset = load_dataset(
    "json",
    data_files={"train": "/content/drive/My Drive/Colab Notebooks/Data/train_bank_intent.jsonl",
                "test": "/content/drive/My Drive/Colab Notebooks/Data/test_bank_intent.jsonl"}
)
train_hf = dataset["train"]
test_hf = dataset["test"]

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [7]:
# Data type of train_hf:
print("Data type of train_hf ", type(train_hf))
print("Shape of trainning set:", train_hf.num_rows, train_hf.num_columns)
print("Shape of testing set:", test_hf.num_rows, test_hf.num_columns)
print("Sort label in train set ", np.sort(list(set(train_hf["label"]))))
print("Count total label in train set ", len(set(train_hf["label"])))

print("Sort label in test set ", np.sort(list(set(test_hf["label"]))))
print("Count total label in test set ", len(set(test_hf["label"])))

print("Count label text in train set ", len(set(train_hf["label_text"])))
print("Count label text in test set ", len(set(test_hf["label_text"])))


# inspect labels
print("Features in train set ",train_hf.features)

Data type of train_hf  <class 'datasets.arrow_dataset.Dataset'>
Shape of trainning set: 10003 3
Shape of testing set: 3080 3
Sort label in train set  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76]
Count total label in train set  77
Sort label in test set  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76]
Count total label in test set  77
Count label text in train set  77
Count label text in test set  77
Features in train set  {'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string')}


In [ ]:
# create a copy version of features of the training set
# new_features has type datasets.Features
new_features = train_hf.features.copy()

label_uni_val = set(train_hf["label"])
# make sure the label column has continuous values
if label_uni_val != set(range(min(label_uni_val),max(label_uni_val) + 1)):
    raise ValueError(f"Values of label column are not continuous numbers. Dataset is invalid")

num_class = len(set(train_hf["label"]))
new_features["label"] = ClassLabel(num_classes=num_class)

# convert data type of "label" column from int to ClassLabel type to define this column as categorical target
train_hf = train_hf.cast(new_features)

# split validation set from the training set
splits = train_hf.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
dataset = DatasetDict({
    "train": splits["train"],
    "validation": splits["test"],
    "test": dataset["test"]
})

Casting the dataset:   0%|          | 0/10003 [00:00<?, ? examples/s]

In [ ]:
# mapping from label id to label text and save in id2label dict
# mapping from label text to label id and save in label2id dict
def labelmapping(hf: Dataset, numUniqueLabel):
    id2label = {}
    label2id = {}
    # # for index, rec in df.iterrows():
    # for rec in df:
    #     lab = int(rec["label"])
    #     txt = rec["label_text"]
    #     # lab = int(getattr(rec, "label"))
    #     # txt = getattr(rec, "label_text")

    #     # throw error if one label is mapped to two different text
    #     if lab in id2label and id2label[lab] != txt:
    #         raise ValueError(f"Value {lab} is mapped to both {id2label[lab]} and {txt}")

    #     # throw error if one text is mapped to two different labels
    #     if txt in label2id and label2id[txt] != lab:
    #         raise ValueError(f"Value {txt} is mapped to both {label2id[txt]} and {lab}")

    #     id2label[lab] = txt
    #     label2id[txt] = lab

    # # check if there is any missing or redundant index in the id2label dict
    # expect = set(range(numUniqueLabel))
    # missing = expect - set(id2label.keys())
    # extra = set(id2label.keys()) - expect

    # # If there are items in set missing or extra then throw error
    # if missing or extra:
    #     raise ValueError(f"There are missing indices {missing} or extra indices {extra} in the mapping from index to text")

    # select a temp df that include 2 columns and remove all duplicate rows
    temp = hf.select_columns(["label", "label_text"]).to_pandas().drop_duplicates()

    # check if one label is mapped to two different label text
    # label_count is a pandas Series
    label_count = temp.groupby(["label"])["label_text"].nunique()
    if (label_count > 1).any():
        # keep entries where label_count > 1
        # for ex: [4,2,1] --> [4,2]
        bad_rec = label_count[label_count > 1]
        raise ValueError(f"Label mapped to multiple label texts:\n {bad_rec}")

    # check if one label text is mapped to two different labels
    text_count = temp.groupby("label_text")["label"].nunique()
    if (text_count > 1).any():
        bad_rec = text_count[text_count > 1]
        raise ValueError(f"Label text mapped to multiple labels:\n{bad_rec}")

    # build mapping
    # dict(zip(...)): pair the 2 columns and then build a dict for these 2 columns
    id2label = dict(zip(temp["label"], temp["label_text"]))
    label2id = dict(zip(temp["label_text"], temp["label"]))

    return id2label, label2id

In [ ]:
# get label mapping of training/test set
# numUniqueLabel = len(set(train_hf["label"]))
train_id2label, train_label2id = labelmapping(train_hf, num_class)
test_id2label, test_label2id = labelmapping(test_hf, num_class)

# make sure the test set also has the same number of labels as the training set
if not (train_id2label == test_id2label and train_label2id == test_label2id):
    missing_label = set(train_id2label.keys()) - set(test_id2label.keys())
    extra_label = set(test_id2label.keys()) - set(train_id2label.keys())
    missing_text = set(train_label2id.keys()) - set(test_label2id.keys())
    extra_text = set(test_label2id.keys()) - set(train_label2id.keys())

    if missing_label or extra_label:
        raise ValueError(f"Labels in train and text are different which is missing at test {missing_label} and extra at train {extra_label}")

    if missing_text or extra_text:
        raise ValueError(f"Label text in train and text are different which is missing at test {missing_text} and extra at train {extra_text}")

model_checkpoint = 'distilbert-base-uncased'

# generate classification model from model_checkpoint
# load a pretrained encoder (DistilBERT weights)
# Creates a new classification head with size num_labels
# - Classification head takes sentence vector (batch_size x hidden_size) and produce logits: logits = hW + b
# Attaches label metadata (id2label, label2id) for interpretation
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=len(train_id2label.keys()), id2label = train_id2label, label2id = train_label2id
)

# preprocess data
# load tokenizer represents the vocab list and tokenization rules of the pretrained model
# add_prefix_space=True: Add a leading space at the begining of the input text before tokenization
# to separate between tokens
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# create token id representing tokens
def tokenizer_func(examples):
    # extract text, can be a string (a sentence) or a list of string (multiple sentences - batch)
    text = examples["text"]

    # if a text is longer than max_length, truncation will remove tokens from the right
    tokenizer.truncation_side = "right" # default is on right side

    # create integer IDs representing tokens
    tokenizer_inputs = tokenizer(
        text,
        # return_tensors = "np", # returns Numpy arrays instead of Python lists or PyTorch tensors
        truncation=True,
        max_length=128
    )

    # return a dict contains input_ids and attention_mask have size of batch_size x sequence_len
    # attention_mask 1: real tokens, 0: padding tokens
    # For ex:
    # {
    #    "input_ids": array([[  101,  1045,  2572,  2145,  3401, 0],
    #                        [2006,  2026,  4003,  1029,  102, 876 ]]),
    #    "attention_mask": array([[1, 1, 1, 1, 1, 0],
    #                             [1, 1, 1, 1, 1, 1]])
    # }
    # --> batch_size = 2 (2 sentences), sequence_len = 6 (6 tokens)
    return tokenizer_inputs

# add pad token string (ID = 0) into tokenizer's vocab if non exists
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    # After adding pad token, the tokenizer vocab size increases
    # --> We need to add a new row to the embedding matrix (has size vocab_size x hidden size) created inside the pretrained model
    model.resize_token_embeddings(len(tokenizer))

# tokenize training and validation datasets
# tokenizer_func will be applied to many dataset rows
# batch=True: Hugging Face passes multiple examples at once to tokenizer_func to speed up processing time
# The result is a new dataset with additional columns from tokenizer_inputs
tokenized_dataset = dataset.map(tokenizer_func, batched=True)
print("tokenized_dataset ",tokenized_dataset)

cols = ["input_ids", "attention_mask", "label"]

# format columns as PyTorch tensors data type so that model can compute loss and gradients (for training set)
# and compute loss, forward pass/logits on the eval and test data
tokenized_dataset["train"].set_format("torch", columns=cols)
tokenized_dataset["validation"].set_format("torch", columns=cols)
tokenized_dataset["test"].set_format("torch",columns=cols)
print("tokenized_dataset after formating",tokenized_dataset)

# create data collator to dynamically fill pad token at the short sequences at batch time
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# import accuracy evaluation metric
accuracy = evaluate.load("accuracy")

# define an evaluation func to pass into trainer later
# def compute_metrics(p):
def compute_metrics(eval_pred):
    # logits is a matrix has shape batch_size x num_labels)
    logits = eval_pred.predictions
    # labels (shape batch_size)
    labels = eval_pred.label_ids
    # convert logit to prediction row-wise (axis=1) which is the index label that has max logit
    preds = np.argmax(logits, axis=1)
    # compute and return accuracy metric
    return accuracy.compute(predictions=preds, references=labels)

# task_type = "SEQ_CLS": The task LoRA is doing is sequence classification (intent classification, sentiment,...)
# Other task types can be "CAUSAL_LM" (text generation), "TOKEN_CLS" (token classification), "SEQ_2_SEQ_LM" (translation, summarization)
peft_config = LoraConfig(task_type="SEQ_CLS",
                         r=4, #LoRA rank
                        # Scales the LoRA update before adding it to the frozen weight
                        # Weff = W + (alpha/r).BA
                         lora_alpha=32,
                        # Dropout is applied to activation func in the LoRA dapter path (output=Wx+B(Dropout(Ax))) during the forward pass in training
                        # It randomly sets a fraction (e.g., 1%) of activation values to zero to inject noise so that model does not rely on any singal feature too much
                         lora_dropout=0.01,
                        # Inject LoRA adapters to query projection layer in self-attention
                        # Because query vectors control what a token attends to --> a very effective layer for adaptation
                         target_modules=['q_lin']
                         )

# Frozen base model's weights, only LoRA params are trainable
model = get_peft_model(model, peft_config)
# print out information about which params are trainable
model.print_trainable_parameters()

# hyperparameters
lr = 1e-3 # size of optimization step
batch_size = 4 # number of examples processed per optimization step
num_epochs = 10 # number of times model runs through training data

# setup output folder
ts = time.strftime("%Y%m%d_%H%M%S")

RUN_DIR = f"distilbert-intent-lora-run_{ts}"
FINAL_DIR = f"distilbert-intent-lora-FINAL_{ts}"

print("RUN_DIR:", RUN_DIR)
print("FINAL_DIR:", FINAL_DIR)

# define training arguments
training_args = TrainingArguments(
    # directory where Trainer save trained model, training logs, optimizer states
    # output_dir = model_checkpoint + "-lora-text-classification",
    output_dir = RUN_DIR,
    # step size used by optimizer when updating parameters, apply to only LoRA params (A&B)
    learning_rate=lr,
    # num of samples processed per device (GPU/CPU) in one forward/backward pass
    # For 1 device: num of steps per epoch = num of samples/per_device_batch_size
    # For 2 device: num of steps per epoch = num of samples/2*per_device_batch_size
    per_device_train_batch_size=batch_size,
    # num of samples processed per device in one forward pass during evaluation process
    # For 2 device: num of samples processed in one forward pass = 2*4 = 8
    # Evaluation runs in num of samples/2*per_device_eval_batch_size steps
    per_device_eval_batch_size=batch_size,

    num_train_epochs=num_epochs,
    # L2 regularization, penalize large weights (discourages Δ𝑊=𝐵𝐴 from becoming too large) to prevent overfitting
    # because large weights --> model's output change a lot for a small change in inputs
    # --> model learns noise, not learn the general patterns --> overfitting issue
    weight_decay=0.01,

    # perform evaluation at the end of each epoch
    eval_strategy="epoch",
    # save one model check point at the end of each epoch
    save_strategy="epoch",
    # After training finishes, Trainer reloads the best checkpoint into trainer.model
    # Best checkpoint is identified based on lowest eval loss by default
    load_best_model_at_end=True,
)

# create Trainer object
# So we provide dataset (inputs + labels).
# Trainer handles below tasks:
# for epoch:
#   for batch:
#   - Forward pass (compute logits using W+ΔW)
#   - Loss computation
#   - Backward pass (compute gradient for low rank matrices A and B)
#   - Update low rank matrices A & B
#   - zero_grad (remove gradient)
#   Evaluation (use lasted trained weights A,B to compute logits=f(x; θ)) of current epoch

# The Trainer object will produce the below output
# SequenceClassifierOutput(
#     loss=..., # this loss is a scalar, it's loss for each batch and it goes to backward pass during training
#     logits=Tensor[batch_size, num_labels] --> it's passed into compute_metrics during evaluation process
# )
trainer = Trainer(
    model=model,
    args=training_args,
    # set input to base model which is a matrix size batch_size x sequence_len/sentence_len
    # each item in these datasets looks like:
    # {
    # "input_ids": Tensor[seq_len],
    # "attention_mask": Tensor[seq_len],
    # "label": Tensor[] --> This is true index label
    # }
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator, #this will dynamically pad examples
    # during evaluation, model passes logits (shape batch_size x num_labels) and labels (shape batch_size)
    # to compute_metrics where we finally can see the output(predictions) and accuracy
    compute_metrics=compute_metrics, #evaluate model using compute_metrics
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

tokenized_dataset  DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 8002
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 3080
    })
})
tokenized_dataset after formating DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 8002
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 3080
    })
})


trainable params: 686,669 || all params: 67,699,354 || trainable%: 1.0143
RUN_DIR: distilbert-intent-lora-run_20251230_125611
FINAL_DIR: distilbert-intent-lora-FINAL_20251230_125611


/tmp/ipython-input-3217156597.py:190: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
# train model
trainer.train()
trainer.evaluate()
trainer.predict(tokenized_dataset["test"])

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.863200,0.643831,0.815592
2,0.660700,0.649858,0.842079
3,0.525800,0.556026,0.877561
4,0.451900,0.612277,0.882059
5,0.403000,0.662142,0.878561
6,0.303300,0.583784,0.893053
7,0.227500,0.566208,0.909045
8,0.154000,0.604483,0.908046
9,0.115300,0.589628,0.909045
10,0.099300,0.578514,0.907046


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


PredictionOutput(predictions=array([[-11.105642, -22.572392, -21.226633, ..., -14.125011, -30.58411 ,
        -29.81068 ],
       [-16.837435, -26.569187, -24.4528  , ..., -22.302149, -24.352303,
        -24.952103],
       [-17.501558, -26.602276, -26.49421 , ..., -20.021278, -26.443214,
        -26.444702],
       ...,
       [-26.854538, -20.120478, -18.95966 , ..., -20.439741, -33.83795 ,
        -18.316517],
       [-23.024193, -19.952341, -18.737854, ..., -20.377617, -28.744766,
        -17.080502],
       [-23.767311, -24.874397, -14.368664, ..., -23.498198, -38.237278,
        -21.345848]], dtype=float32), label_ids=array([11, 11, 11, ..., 24, 24, 24]), metrics={'test_loss': 0.5486382246017456, 'test_accuracy': 0.8724025974025974, 'test_runtime': 151.0778, 'test_samples_per_second': 20.387, 'test_steps_per_second': 5.097})

In [12]:
# output_dir = model_checkpoint + "-lora-text-classification"
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

('distilbert-intent-lora-FINAL_20251230_125611/tokenizer_config.json',
 'distilbert-intent-lora-FINAL_20251230_125611/special_tokens_map.json',
 'distilbert-intent-lora-FINAL_20251230_125611/vocab.txt',
 'distilbert-intent-lora-FINAL_20251230_125611/added_tokens.json',
 'distilbert-intent-lora-FINAL_20251230_125611/tokenizer.json')

In [13]:
# # Copy the whole folder into Drive
# import shutil, os

# src = "/content/distilbert-base-uncased-lora-text-classification"
# dst = "/content/drive/My Drive/Colab Notebooks/distilbert-base-uncased-lora-text-classification-retrain"

# # Remove destination if it already exists to avoid merge/conflicts
# if os.path.exists(dst):
#     shutil.rmtree(dst)

# shutil.copytree(src, dst)

# print("Copied to:", dst)
# print("Drive folder files:", sorted(os.listdir(dst))[:30])


In [14]:
# import os

# output_dir = "/content/drive/My Drive/Colab Notebooks/distilbert-base-uncased-lora-text-classification"
# print("Exists:", os.path.exists(output_dir))
# print("Files:", sorted(os.listdir(output_dir)))


In [15]:

# # find the best checkpoint to load for deployment
# import os

# base_dir = "/content/drive/My Drive/Colab Notebooks/distilbert-base-uncased-lora-text-classification-retrain"

# ckpts = sorted(
#     [d for d in os.listdir(base_dir) if d.startswith("checkpoint-")],
#     key=lambda x: int(x.split("-")[1])
# )

# print("All checkpoints:", ckpts)
# best_ckpt = os.path.join(base_dir, ckpts[-1])
# print("Using checkpoint:", best_ckpt)



In [16]:
import json, os

# Make JSON keys strings (safe for JSON)
with open(os.path.join(FINAL_DIR, "id2label.json"), "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in train_id2label.items()}, f, ensure_ascii=False, indent=2)

with open(os.path.join(FINAL_DIR, "label2id.json"), "w", encoding="utf-8") as f:
    json.dump(train_label2id, f, ensure_ascii=False, indent=2)

# Also save num_labels explicitly (nice for debugging)
with open(os.path.join(FINAL_DIR, "num_labels.json"), "w", encoding="utf-8") as f:
    json.dump({"num_labels": len(train_id2label)}, f, indent=2)


In [17]:
from google.colab import drive
import os

drive.mount("/content/drive")

print("Drive exists:", os.path.exists("/content/drive/MyDrive"))
print("Top-level in MyDrive:", os.listdir("/content/drive/MyDrive")[:20])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive exists: True
Top-level in MyDrive: ['Data Science', 'Colab Notebooks', 'KinKenAndMe', 'RESUME - NGOC VAN - UAB.pdf', 'I-94_Form.pdf', 'on-campus employment verification letter.pdf', 'SSNApplication.pdf']


In [18]:
import os, shutil

# Example: FINAL_DIR created earlier in /content
# FINAL_DIR = "/content/distilbert-intent-lora-FINAL_YYYYMMDD_HHMMSS"
print("FINAL_DIR exists:", os.path.exists(FINAL_DIR))
print("FINAL_DIR files:", os.listdir(FINAL_DIR))

dst = f"/content/drive/MyDrive/Colab Notebooks/{os.path.basename(FINAL_DIR)}"

# Remove destination first to avoid merges
if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(FINAL_DIR, dst)

print("Copied to:", dst)
print("Drive folder now contains:", os.listdir(dst))


FINAL_DIR exists: True
FINAL_DIR files: ['num_labels.json', 'vocab.txt', 'tokenizer.json', 'README.md', 'tokenizer_config.json', 'id2label.json', 'label2id.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'adapter_config.json']
Copied to: /content/drive/MyDrive/Colab Notebooks/distilbert-intent-lora-FINAL_20251230_125611
Drive folder now contains: ['num_labels.json', 'vocab.txt', 'tokenizer.json', 'README.md', 'tokenizer_config.json', 'id2label.json', 'label2id.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'adapter_config.json']


In [19]:
import os

for name in sorted(os.listdir("/content")):
    print(name)


.config
distilbert-intent-lora-FINAL_20251230_125611
distilbert-intent-lora-run_20251230_125611
drive
sample_data
wandb
